<a href="https://colab.research.google.com/github/postnicov/RowingSPb/blob/main/Rowing_Analysis_Upload_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rowing data analysis — upload workflow

The notebook keeps unwrapped and wrapped Hilbert phases as separate, explicitly named objects. Unwrapped phases are calculated with `numpy.unwrap`; wrapped phases are obtained only by mapping an unwrapped phase or phase difference to the interval from `-pi` to `pi`.

In [ ]:
#@title 1. Install dependencies and import libraries
import importlib.util
import subprocess
import sys

def ensure_package(package, import_name=None):
    name = import_name or package
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

ensure_package('EMD-signal', 'PyEMD')
ensure_package('openpyxl')

import io
import re
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from scipy.signal import hilbert
from PyEMD import EMD
from IPython.display import Markdown, display
from google.colab import files

pio.renderers.default = 'colab'
ALL_FIGURES = []
print('Dependencies are ready.')

In [ ]:
#@title 2. Graphical settings
FONT_FAMILY = 'serif'
FONT_SIZE = 14
LINE_WIDTH = 1.5

LAYOUT_DEFAULTS = dict(
    font=dict(family=FONT_FAMILY, size=FONT_SIZE),
    legend=dict(font=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
    xaxis=dict(title_font=dict(family=FONT_FAMILY, size=FONT_SIZE), tickfont=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
    yaxis=dict(title_font=dict(family=FONT_FAMILY, size=FONT_SIZE), tickfont=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
)

def register_figure(fig):
    ALL_FIGURES.append(fig)
    fig.show()

def safe_filename(text, fallback='output'):
    text = str(text).strip()
    text = re.sub(r'<[^>]+>', '', text)
    text = text.replace('—', '-').replace('–', '-').replace('−', '-')
    text = re.sub(r'[\\/:*?"<>|]+', '_', text)
    text = re.sub(r'\s+', '_', text)
    text = re.sub(r'_+', '_', text)
    return text[:180].strip('._-') or fallback

print(f'Font: {FONT_FAMILY}; size: {FONT_SIZE}; line width: {LINE_WIDTH}')

In [ ]:
#@title 3. Upload and parse a tab-delimited .txt file
print('Select a tab-delimited rowing-data .txt file.')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file was uploaded. Re-run this cell and select a .txt file.')

DATA_FILENAME, uploaded_value = next(iter(uploaded.items()))
if isinstance(uploaded_value, (bytes, bytearray)):
    raw_bytes = bytes(uploaded_value)
elif hasattr(uploaded_value, 'read'):
    raw_bytes = uploaded_value.read()
elif isinstance(uploaded_value, str):
    possible_path = Path(uploaded_value)
    raw_bytes = possible_path.read_bytes() if possible_path.is_file() else uploaded_value.encode('utf-8')
else:
    raise TypeError(f'Unsupported upload payload type: {type(uploaded_value).__name__}')
if not raw_bytes:
    raise ValueError(f'Uploaded file {DATA_FILENAME!r} is empty.')

INPUT_STEM = safe_filename(Path(str(DATA_FILENAME)).stem, fallback='rowing_data')
try:
    df_full = pd.read_csv(io.BytesIO(raw_bytes), sep=None, header=0, engine='python', encoding='utf-8-sig')
except UnicodeDecodeError:
    df_full = pd.read_csv(io.BytesIO(raw_bytes), sep=None, header=0, engine='python', encoding='cp1251')

if len(df_full.columns) == 1 and str(df_full.columns[0]).strip() in {'{', '['}:
    preview = raw_bytes[:250].decode('utf-8', errors='replace')
    raise ValueError('The uploaded content appears to be JSON, not a rowing-data .txt file. ' + preview)

df_full = df_full.loc[:, ~df_full.columns.astype(str).str.match(r'^Unnamed', na=False)]
df_full = df_full.dropna(axis=1, how='all')

def normalize_header(header):
    return re.sub(r'\s+', '', str(header).replace('\ufeff', '')).upper()

df_full.columns = [normalize_header(c) for c in df_full.columns]
print(f'Loaded: {DATA_FILENAME}')
print(f'Raw shape: {len(df_full)} rows x {len(df_full.columns)} columns')
print('Parsed headers:', list(df_full.columns))

boat_header_map = {'N': 'N', 'T': 'T', 'AS': 'As', 'VS': 'Vs', 'GPSD': 'GPSd'}
missing_boat = [source for source in boat_header_map if source not in df_full.columns]
if missing_boat:
    raise ValueError(f'Required boat-level channel(s) are absent: {missing_boat}. Parsed headers: {list(df_full.columns)}')

rower_pattern = re.compile(r'^([AVHS])(\d+)$')
rower_numbers = sorted({int(m.group(2)) for c in df_full.columns if (m := rower_pattern.fullmatch(c))})
if not rower_numbers:
    raise ValueError(f'No A#/V#/H#/S# rower channels detected. Parsed headers: {list(df_full.columns)}')
N_ROWERS = len(rower_numbers)

source_columns = list(boat_header_map) + [
    f'{letter}{number}'
    for number in rower_numbers
    for letter in ('A', 'V', 'H', 'S')
    if f'{letter}{number}' in df_full.columns
]
output_columns = list(boat_header_map.values()) + source_columns[len(boat_header_map):]
df_raw = df_full.loc[:, source_columns].copy()
df_raw.columns = output_columns
df_raw = df_raw.apply(pd.to_numeric, errors='coerce')

invalid = [c for c in ('N', 'T', 'As', 'Vs', 'GPSd') if df_raw[c].isna().all()]
if invalid:
    raise ValueError(f'Required channel(s) contain no numeric values: {invalid}')

DATA_TMIN = float(df_raw['T'].min())
DATA_TMAX = float(df_raw['T'].max())
print(f'Detected {N_ROWERS} rower(s): {rower_numbers}')
print(f'Available time range: {DATA_TMIN:.6f} to {DATA_TMAX:.6f} s')
df_raw.head(3)

In [ ]:
#@title 4. Column descriptions
display(Markdown(f'''
## Column descriptions

- **N** — sample number
- **T** — time, s
- **As** — boat acceleration, m/s²
- **Vs** — boat speed (GPS), m/s
- **GPSd** — distance covered (GPS), m
- **A#** — horizontal oar angle, degrees (# = rower number counted from the stern; stroke = 1)
- **V#** — vertical oar angle, degrees
- **H#** — handle force, N
- **S#** — seat position, m

**Detected rowers:** {', '.join(map(str, rower_numbers))} (total: {N_ROWERS})
**Available time range:** {DATA_TMIN:.6f} to {DATA_TMAX:.6f} s
'''))

In [ ]:
#@title 5. Plot raw boat acceleration vs time
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_raw['T'], y=df_raw['As'], mode='lines', name='As', line=dict(width=LINE_WIDTH)))
fig.update_layout(title='Raw data — As vs T', xaxis_title='T (s)', yaxis_title='As (m/s²)', **LAYOUT_DEFAULTS)
register_figure(fig)

In [ ]:
#@title 6. Set analysis time window from available data

#@markdown By default, use the full time range detected in the uploaded file.
use_full_data_range = True #@param {type:"boolean"}

#@markdown Manual limits are used only when `use_full_data_range` is unchecked.
manual_Tmin = 0.0 #@param {type:"number"}
manual_Tmax = 1.0 #@param {type:"number"}

if use_full_data_range:
    Tmin = DATA_TMIN
    Tmax = DATA_TMAX
    range_mode = "Automatic: full uploaded-data range"
else:
    Tmin = float(manual_Tmin)
    Tmax = float(manual_Tmax)
    range_mode = "Manual"

if Tmin < DATA_TMIN or Tmax > DATA_TMAX:
    raise ValueError(
        f"Time limits must lie within the available range: "
        f"{DATA_TMIN:.6f} to {DATA_TMAX:.6f} s."
    )

if Tmin >= Tmax:
    raise ValueError("Tmin must be smaller than Tmax.")

print(f"Available data range: {DATA_TMIN:.6f} to {DATA_TMAX:.6f} s")
print(f"Selected range mode:  {range_mode}")
print(f"Analysis window:      {Tmin:.6f} to {Tmax:.6f} s")

In [ ]:
#@title 7. Trim data and redefine time relative to Tmin
df = df_raw.loc[(df_raw['T'] >= Tmin) & (df_raw['T'] <= Tmax)].copy().reset_index(drop=True)
if len(df) < 3:
    raise ValueError('The selected time window contains fewer than three rows.')
df['T'] = df['T'] - Tmin
print(f'Rows after trimming: {len(df)}')
print(f'Processed time range: {df["T"].min():.6f} to {df["T"].max():.6f} s')

In [ ]:
#@title 8. Empirical Mode Decomposition for all signal columns
signal_cols = [c for c in df.columns if c not in ('N', 'T')]
T_arr = df['T'].to_numpy(dtype=float)
emd = EMD()
imfs, trends = {}, {}
for col in signal_cols:
    sig = df[col].to_numpy(dtype=float)
    finite = np.isfinite(sig)
    if finite.sum() < 3:
        print(f'{col}: skipped; fewer than 3 finite values')
        continue
    if not finite.all():
        sig = np.interp(T_arr, T_arr[finite], sig[finite])
    result = emd.emd(sig, T_arr)
    imfs[col] = result
    trends[col] = result[-1] if result.size else np.full_like(sig, np.nanmean(sig), dtype=float)
    print(f'{col}: {result.shape[0]} IMFs decomposed')
signal_cols = [c for c in signal_cols if c in trends]
if not signal_cols:
    raise RuntimeError('No analysable signal columns remain after validation.')

In [ ]:
#@title 9. Group rower signals and plot EMD trends
def rower_number(colname):
    match = re.search(r'(\d+)$', colname)
    return int(match.group(1)) if match else None

groups = defaultdict(list)
for col in signal_cols:
    match = re.fullmatch(r'([AVHS])(\d+)', col)
    if match:
        groups[match.group(1)].append(col)
for letter in groups:
    groups[letter] = sorted(groups[letter], key=rower_number)

for letter, members in sorted(groups.items()):
    fig = go.Figure()
    for col in members:
        fig.add_trace(go.Scatter(x=T_arr, y=trends[col], mode='lines', name=col, line=dict(width=LINE_WIDTH)))
    fig.update_layout(title=f'Trend IMF — group {letter}', xaxis_title='T (s)', yaxis_title=f'{letter} trend', **LAYOUT_DEFAULTS)
    register_figure(fig)

In [ ]:
#@title 10. Detrend signals
detrended = {col: df[col].to_numpy(dtype=float) - trends[col] for col in signal_cols}
print('Detrending complete.')

In [ ]:
#@title 11. Plot detrended rower signals
for letter, members in sorted(groups.items()):
    fig = go.Figure()
    for col in members:
        fig.add_trace(go.Scatter(x=T_arr, y=detrended[col], mode='lines', name=col, line=dict(width=LINE_WIDTH)))
    fig.update_layout(title=f'Detrended signals — group {letter}', xaxis_title='T (s)', yaxis_title=f'{letter} detrended', **LAYOUT_DEFAULTS)
    register_figure(fig)

In [ ]:
#@title 12. Calculate unwrapped and wrapped Hilbert phases separately
# Definitions:
# phase_principal: principal Hilbert phase in (-pi, pi].
# phase_unwrapped: np.unwrap(phase_principal), preserving phase accumulation.
# phase_wrapped: phase_unwrapped mapped to [-pi, pi); used only for wrapped plots/exports.
def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

hilbert_signals = {}
phase_principal = {}
phase_unwrapped = {}
phase_wrapped = {}

for letter, members in sorted(groups.items()):
    hilbert_signals[letter] = {}
    phase_principal[letter] = {}
    phase_unwrapped[letter] = {}
    phase_wrapped[letter] = {}
    for col in members:
        analytic_signal = hilbert(np.asarray(detrended[col], dtype=float))
        principal = np.angle(analytic_signal)
        unwrapped = np.unwrap(principal)
        wrapped = wrap_to_pi(unwrapped)
        hilbert_signals[letter][col] = analytic_signal
        phase_principal[letter][col] = principal
        phase_unwrapped[letter][col] = unwrapped
        phase_wrapped[letter][col] = wrapped
    print(f'{letter}: separate principal, unwrapped, and wrapped phases computed for {len(members)} rower(s)')

In [ ]:
#@title 13. Plot unwrapped Hilbert phase for every rower
for letter, members in sorted(groups.items()):
    fig = go.Figure()
    for col in members:
        fig.add_trace(go.Scatter(x=T_arr, y=phase_unwrapped[letter][col], mode='lines', name=col, line=dict(width=LINE_WIDTH)))
    fig.update_layout(title=f'Unwrapped Hilbert phase — group {letter}', xaxis_title='T (s)', yaxis_title='Unwrapped phase (rad)', **LAYOUT_DEFAULTS)
    register_figure(fig)

In [ ]:
#@title 14. Plot wrapped Hilbert phase for every rower
for letter, members in sorted(groups.items()):
    fig = go.Figure()
    for col in members:
        fig.add_trace(go.Scatter(x=T_arr, y=phase_wrapped[letter][col], mode='lines', name=col, line=dict(width=LINE_WIDTH)))
    fig.update_layout(title=f'Wrapped Hilbert phase — group {letter}', xaxis_title='T (s)', yaxis_title='Wrapped phase (rad)', **LAYOUT_DEFAULTS)
    register_figure(fig)

In [ ]:
#@title 15. Calculate phase differences to stroke rower separately
# Unwrapped difference = unwrapped(rower) - unwrapped(stroke).
# Wrapped difference = wrap_to_pi(unwrapped difference).
phase_difference_to_stroke_unwrapped = {}
phase_difference_to_stroke_wrapped = {}

for letter, members in sorted(groups.items()):
    ref_col = f'{letter}1'
    phase_difference_to_stroke_unwrapped[letter] = {}
    phase_difference_to_stroke_wrapped[letter] = {}
    if ref_col not in members:
        print(f'{letter}: {ref_col} is absent; no stroke-reference differences calculated.')
        continue
    for col in members:
        if col == ref_col:
            continue
        difference_unwrapped = phase_unwrapped[letter][col] - phase_unwrapped[letter][ref_col]
        phase_difference_to_stroke_unwrapped[letter][col] = difference_unwrapped
        phase_difference_to_stroke_wrapped[letter][col] = wrap_to_pi(difference_unwrapped)
    print(f'{letter}: stroke-reference phase differences calculated')

In [ ]:
#@title 16. Plot unwrapped phase differences relative to stroke rower
for letter, members in sorted(groups.items()):
    ref_col = f'{letter}1'
    differences = phase_difference_to_stroke_unwrapped.get(letter, {})
    if not differences:
        continue
    fig = go.Figure()
    for col, values in differences.items():
        fig.add_trace(go.Scatter(x=T_arr, y=values, mode='lines', name=f'{col} − {ref_col}', line=dict(width=LINE_WIDTH)))
    fig.update_layout(title=f'Unwrapped Hilbert phase difference to stroke rower — group {letter}', xaxis_title='T (s)', yaxis_title='Unwrapped phase difference (rad)', **LAYOUT_DEFAULTS)
    register_figure(fig)

In [ ]:
#@title 17. Plot wrapped phase differences relative to stroke rower
for letter, members in sorted(groups.items()):
    ref_col = f'{letter}1'
    differences = phase_difference_to_stroke_wrapped.get(letter, {})
    if not differences:
        continue
    fig = go.Figure()
    for col, values in differences.items():
        fig.add_trace(go.Scatter(x=T_arr, y=values, mode='lines', name=f'{col} − {ref_col}', line=dict(width=LINE_WIDTH)))
    fig.update_layout(title=f'Wrapped Hilbert phase difference to stroke rower — group {letter}', xaxis_title='T (s)', yaxis_title='Wrapped phase difference (rad)', **LAYOUT_DEFAULTS)
    register_figure(fig)

In [ ]:
#@title 18. Calculate phase differences between successive rowers separately
phase_difference_successive_unwrapped = {}
phase_difference_successive_wrapped = {}

for letter, members in sorted(groups.items()):
    phase_difference_successive_unwrapped[letter] = {}
    phase_difference_successive_wrapped[letter] = {}
    for previous, current in zip(members[:-1], members[1:]):
        difference_unwrapped = phase_unwrapped[letter][current] - phase_unwrapped[letter][previous]
        label = f'{current} − {previous}'
        phase_difference_successive_unwrapped[letter][label] = difference_unwrapped
        phase_difference_successive_wrapped[letter][label] = wrap_to_pi(difference_unwrapped)
    if len(members) >= 2:
        print(f'{letter}: successive-rower phase differences calculated')

In [ ]:
#@title 19. Plot unwrapped phase differences between successive rowers
for letter, differences in sorted(phase_difference_successive_unwrapped.items()):
    if not differences:
        continue
    fig = go.Figure()
    for label, values in differences.items():
        fig.add_trace(go.Scatter(x=T_arr, y=values, mode='lines', name=label, line=dict(width=LINE_WIDTH)))
    fig.update_layout(title=f'Unwrapped Hilbert phase difference between successive rowers — group {letter}', xaxis_title='T (s)', yaxis_title='Unwrapped phase difference (rad)', **LAYOUT_DEFAULTS)
    register_figure(fig)

In [ ]:
#@title 20. Plot wrapped phase differences between successive rowers
for letter, differences in sorted(phase_difference_successive_wrapped.items()):
    if not differences:
        continue
    fig = go.Figure()
    for label, values in differences.items():
        fig.add_trace(go.Scatter(x=T_arr, y=values, mode='lines', name=label, line=dict(width=LINE_WIDTH)))
    fig.update_layout(title=f'Wrapped Hilbert phase difference between successive rowers — group {letter}', xaxis_title='T (s)', yaxis_title='Wrapped phase difference (rad)', **LAYOUT_DEFAULTS)
    register_figure(fig)

In [ ]:
#@title 21. Export all generated Plotly figures to HTML and pack into ZIP
if not ALL_FIGURES:
    raise RuntimeError('No Plotly figures have been registered. Run the plotting cells before exporting.')
EXPORT_DIR = Path(f'{INPUT_STEM}_plotly_html')
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

saved_files = []
for index, fig in enumerate(ALL_FIGURES, start=1):
    title = fig.layout.title.text if fig.layout.title.text else f'figure_{index}'
    filename = f'{INPUT_STEM}_{index:02d}_{safe_filename(title, fallback=f"figure_{index}")}.html'
    path = EXPORT_DIR / filename
    fig.write_html(str(path), include_plotlyjs='cdn', full_html=True)
    saved_files.append(path)

zip_name = f'{INPUT_STEM}_plotly_figures_html.zip'
with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in saved_files:
        archive.write(path, arcname=path.name)
print(f'Exported {len(saved_files)} Plotly HTML files to {zip_name}.')
files.download(zip_name)

In [ ]:
#@title 22. Export processed data to Excel
sheet_original = df.copy()
sheet_trends = pd.DataFrame({'N': df['N'].to_numpy(), 'T': T_arr})
sheet_detrended = pd.DataFrame({'N': df['N'].to_numpy(), 'T': T_arr})
sheet_hilbert_real = pd.DataFrame({'N': df['N'].to_numpy(), 'T': T_arr})
sheet_hilbert_imag = pd.DataFrame({'N': df['N'].to_numpy(), 'T': T_arr})
sheet_phase_principal = pd.DataFrame({'N': df['N'].to_numpy(), 'T': T_arr})
sheet_phase_unwrapped = pd.DataFrame({'N': df['N'].to_numpy(), 'T': T_arr})
sheet_phase_wrapped = pd.DataFrame({'N': df['N'].to_numpy(), 'T': T_arr})

for col in signal_cols:
    sheet_trends[col] = trends[col]
    sheet_detrended[col] = detrended[col]

for letter, members in groups.items():
    for col in members:
        sheet_hilbert_real[col] = np.real(hilbert_signals[letter][col])
        sheet_hilbert_imag[col] = np.imag(hilbert_signals[letter][col])
        sheet_phase_principal[col] = phase_principal[letter][col]
        sheet_phase_unwrapped[col] = phase_unwrapped[letter][col]
        sheet_phase_wrapped[col] = phase_wrapped[letter][col]

xlsx_name = f'{INPUT_STEM}_rowing_data_analysis.xlsx'
with pd.ExcelWriter(xlsx_name, engine='openpyxl') as writer:
    sheet_original.to_excel(writer, sheet_name='Original data', index=False)
    sheet_trends.to_excel(writer, sheet_name='Trends', index=False)
    sheet_detrended.to_excel(writer, sheet_name='Detrended signals', index=False)
    sheet_hilbert_real.to_excel(writer, sheet_name='Hilbert real', index=False)
    sheet_hilbert_imag.to_excel(writer, sheet_name='Hilbert imaginary', index=False)
    sheet_phase_principal.to_excel(writer, sheet_name='Principal phases', index=False)
    sheet_phase_unwrapped.to_excel(writer, sheet_name='Unwrapped phases', index=False)
    sheet_phase_wrapped.to_excel(writer, sheet_name='Wrapped phases', index=False)
print(f'Saved {xlsx_name}: {len(df)} rows, {N_ROWERS} detected rower(s).')
files.download(xlsx_name)